In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# ── Build the CNN ─────────────────────────────────────────────────────────
def create_cnn_model(input_shape, num_classes):
    model = models.Sequential([

        # ── BLOCK 1: First convolution ───────────────────────────────────
        # Conv2D(32, (3,3)):
        #   32 filters, each 3×3 pixels
        #   Each filter learns ONE pattern (e.g. horizontal edge)
        #   32 filters = 32 different patterns learned simultaneously
        #   Output: 32 feature maps, each (26×26) — shrinks by 2 because
        #   a 3×3 filter can't be centred on the border pixels
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),

        # MaxPooling2D(2,2):
        #   Takes every 2×2 block and keeps only the MAX value
        #   Halves spatial dimensions: 26×26 → 13×13
        #   Purpose: reduce size, keep strongest signals, add translation invariance
        #   (if a pattern shifts 1 pixel, pooling still detects it)
        layers.MaxPooling2D((2, 2)),

        # ── BLOCK 2: Second convolution ──────────────────────────────────
        # Now operates on the 32 feature maps from Block 1
        # Learns combinations of lower-level patterns → curves, corners
        # Output: 64 feature maps, each (11×11) → pooling → (5×5)
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # ── BLOCK 3: Third convolution ───────────────────────────────────
        # Deeper patterns: learns high-level shapes (loops, strokes)
        # Output: 64 feature maps, each (3×3)
        layers.Conv2D(64, (3, 3), activation='relu'),

        # ── TRANSITION: Flatten ──────────────────────────────────────────
        # Converts 3D tensor (3×3×64 = 576 values) → flat 1D vector
        # Bridges the spatial CNN layers and the classification Dense layers
        layers.Flatten(),

        # ── CLASSIFICATION HEAD ──────────────────────────────────────────
        # Dense(64, relu): standard fully-connected layer
        # Combines all spatial features to reason about the digit
        layers.Dense(64, activation='relu'),

        # Dense(10, softmax): 10 output neurons, one per digit (0–9)
        # Softmax converts raw scores → probabilities that sum to 1
        # e.g. [0.01, 0.02, 0.90, 0.01, ...] → "this is a 2"
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

# ── Configuration ──────────────────────────────────────────────────────────
input_shape = (28, 28, 1)   # height=28, width=28, channels=1 (greyscale)
                             # RGB images would be (28, 28, 3)
num_classes = 10             # digits 0 through 9

# ── Create, compile, summarise ────────────────────────────────────────────
model = create_cnn_model(input_shape, num_classes)

model.compile(
    optimizer='adam',
    # sparse_categorical_crossentropy: same as categorical crossentropy
    # but labels are integers [0,1,2..9] not one-hot [[1,0,0...],[0,1,0...]]
    # Use this when y_train = [5, 0, 4, 1, 9, ...] (raw digit labels)
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()  # prints layer shapes and parameter counts

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,322 (364.54 KB)

 Trainable params: 93,322 (364.54 KB)

 Non-trainable params: 0 (0.00 B)